In [1]:
import scMPRAforge as scm
from dask.distributed import Client, LocalCluster

2026-01-16 15:42:10.279357: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-16 15:42:10.283796: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [2]:
cluster=LocalCluster(memory_limit='24GB')
client = Client(cluster)

In [3]:
data_root="/nfs/roberts/project/pi_skr2/shared/tabula_data"
path=f"{data_root}/shendure"
name="ortho_primordial_v4"

In [4]:
primordial=scm.ortho.load(client,path,name)
shendure=primordial.training_data

In [5]:
shendure.data

,cell_bc,rep_id,transfection_bc,mpra_bc,cre_class,cre_id,reads_transfection_bc,umis_transfection_bc,reads_mpra_bc,umis_mpra_bc,cell_type,gex_umi,normalized_umis_mpra_bc,cre_id_original
0,A1_GTTACCCAGTTGAAGT-1,A1,GAAAGTGTATTTGGGT,ACGTAACATTATAAT,devCRE,Txndc12_chr4_7978,499,415,0,0,SurfaceEctoderm,3913,0.000000,Txndc12_chr4_7978
1,A1_GTTACCCAGTTGAAGT-1,A1,ACTGACGTCAATCAAT,TGTTTAAGTCAACAA,devCRE,Klf4_chr4_3952,850,678,0,0,SurfaceEctoderm,3913,0.000000,Klf4_chr4_3952
2,A1_GTTACCCAGTTGAAGT-1,A1,GGAGGTGTGCGCGTGG,CAACAACACATTTTA,devCRE,Foxa2_chr2_13840,549,451,0,0,SurfaceEctoderm,3913,0.000000,Foxa2_chr2_13840
3,A1_GTTACCCAGTTGAAGT-1,A1,CGATACCTACTTAATA,TACCTAATGGGAAAG,promoters,reference,413,353,0,0,SurfaceEctoderm,3913,0.000000,minP
4,A1_GTTACCCAGTTGAAGT-1,A1,CCGGGGTTGTAGGTAA,AAATGGTAAAAGGCA,devCRE,Lamc1_chr1_12152,319,258,0,0,SurfaceEctoderm,3913,0.000000,Lamc1_chr1_12152
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
781455,2B2_TTCTGTAAGGCTCTCG-1,2B2,CGACATTGGGGGATGC,CCCACAAAGGCACCA,devCRE,Txndc12_chr4_7961,14,14,0,0,NeuroectodermRostral,522,0.000000,Txndc12_chr4_7961
781456,2B2_TTCTGTAAGGCTCTCG-1,2B2,ATTTAGACACCAATAC,CAGCCCGCCGCCGAG,promoters,ubcP,32,31,196,11,NeuroectodermRostral,522,63.118754,ubcP
781457,2B2_TGGAACTCAATCCTAG-1,2B2,GTTAGATACGCTATGG,ATGGCACATATCTTA,devCRE,Gata4_chr14_5760,21,18,0,0,Mesoderm,968,0.000000,Gata4_chr14_5760
781458,2B2_TGGAACTCAATCCTAG-1,2B2,GATTAATTCGGTCTAC,CTCTATCGTCCTCGT,devCRE,Lamb1_chr12_2289,13,13,0,0,Mesoderm,968,0.000000,Lamb1_chr12_2289


https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html

> You may have noticed that the p-values found here are extremely low. This is due to the statistical test being performed considering each cell as an independent sample. For a more conservative approach you may want to consider “pseudo-bulking” your data by sample (e.g. sc.get.aggregate(adata, by=["sample", "cell_type"], func="sum", layer="counts")) and using a more powerful differential expression tool, like pydeseq2.

Seurat does something similar

https://satijalab.org/seurat/articles/de_vignette

I worry that variable number of cells, barcodes per cell, could mess this up. Ideally we would norm or subsample. But there is no info in the cohen paper suggesting that that's what they did, so I will go with the "default" suggested by the 2x popular libraries.

In [6]:
x=shendure.pseudobulk(subset="Mesoderm",split="cre_id")
x

,rep_id,cre_id,umis_mpra_bc
0,2B1,Bend5_chr4_8172,0
1,2B1,Bend5_chr4_8174,2
2,2B1,Bend5_chr4_8175,117
3,2B1,Bend5_chr4_8179,6
4,2B1,Bend5_chr4_8192,1
...,...,...,...
1057,B2,Txndc12_chr4_7978,24
1058,B2,eef1aP,41978
1059,B2,pgk1P,22935
1060,B2,reference,18


In [7]:
from pydeseq2.utils import load_example_data

In [8]:
counts_df = load_example_data(
    modality="raw_counts",
    dataset="synthetic",
    debug=False,
)

metadata = load_example_data(
    modality="metadata",
    dataset="synthetic",
    debug=False,
)

In [9]:
counts_df

,gene1,gene2,gene3,gene4,gene5,gene6,gene7,gene8,gene9,gene10
sample1,12,21,4,130,18,0,16,54,49,3
sample2,1,44,2,63,11,10,70,32,57,9
sample3,4,4,11,180,21,3,28,34,65,2
sample4,1,10,2,100,44,9,28,16,33,9
sample5,1,11,6,135,16,2,32,29,31,5
...,...,...,...,...,...,...,...,...,...,...
sample96,7,26,3,67,11,4,41,44,54,1
sample97,1,14,3,71,33,5,19,42,25,4
sample98,10,36,2,72,11,2,66,27,16,9
sample99,18,14,3,66,53,11,32,19,79,11


In [10]:
x[["rep_id","cre_id"]]

,rep_id,cre_id
0,2B1,Bend5_chr4_8172
1,2B1,Bend5_chr4_8174
2,2B1,Bend5_chr4_8175
3,2B1,Bend5_chr4_8179
4,2B1,Bend5_chr4_8192
...,...,...
1057,B2,Txndc12_chr4_7978
1058,B2,eef1aP
1059,B2,pgk1P
1060,B2,reference


In [11]:
x[["rep_id","cre_id"]].drop_duplicates()

,rep_id,cre_id
0,2B1,Bend5_chr4_8172
1,2B1,Bend5_chr4_8174
2,2B1,Bend5_chr4_8175
3,2B1,Bend5_chr4_8179
4,2B1,Bend5_chr4_8192
...,...,...
1057,B2,Txndc12_chr4_7978
1058,B2,eef1aP
1059,B2,pgk1P
1060,B2,reference


In [12]:
x.pivot(index='rep_id', columns='cre_id', values='umis_mpra_bc')


cre_id,Bend5_chr4_8172,Bend5_chr4_8174,Bend5_chr4_8175,Bend5_chr4_8179,Bend5_chr4_8192,Bend5_chr4_8199,Bend5_chr4_8201,Btg1_chr10_9578,Btg1_chr10_9588,Btg1_chr10_9593,...,Txndc12_chr4_7951,Txndc12_chr4_7961,Txndc12_chr4_7962,Txndc12_chr4_7971,Txndc12_chr4_7973,Txndc12_chr4_7978,eef1aP,pgk1P,reference,ubcP
rep_id,,,,,,,,,,,,,,,,,,,,,
2B1,0,2,117,6,1,2,2,60,0,19,...,5,16,29,1,79,232,49670,21277,40,8834
2B2,3,0,84,0,1,1,2,110,1,19,...,9,11,39,2,86,62,64976,20216,37,7097
A1,0,0,79,0,0,0,3,75,1,2,...,1,6,14,0,75,48,38982,15844,18,5235
A2,0,2,100,1,0,0,2,131,1,2,...,8,5,24,2,103,40,48551,15459,16,5941
B1,2,5,107,5,1,1,2,58,0,12,...,6,24,19,0,75,27,34171,23340,32,6062
B2,6,3,117,0,0,5,0,58,0,4,...,10,12,4,1,109,24,41978,22935,18,6224


In [13]:
metadata

,condition,group
sample1,A,X
sample2,A,Y
sample3,A,X
sample4,A,Y
sample5,A,X
...,...,...
sample96,B,Y
sample97,B,X
sample98,B,Y
sample99,B,X
